# Baseline 全流程：市场参与者交易行为识别与资金流向分析

> 将 `main.py` 管线拆分为独立单元格，可逐板块运行。

**流程概览：**
1. 扫描股票列表
2. 逐只预处理 + 特征提取 → 即时保存
3. 加载全量特征 → Task1 聚类 / Task2 意图识别
4. 离线评估

## 环境准备：导入模块 & 路径配置

In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np

# 将项目根目录加入 path
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

from modules.preprocess import load_and_preprocess_stock
from modules.features import extract_all_features, save_features, load_all_features
from modules.cluster import run_clustering, assign_patterns, save_pattern_result
from modules.intention import identify_all, save_predict_result

# ============================================================
# 路径配置
# ============================================================
BASE_DIR = os.path.abspath('.')
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw', '100stock')
FEATURES_DIR = os.path.join(BASE_DIR, 'data', 'features')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

print(f"BASE_DIR: {BASE_DIR}")
print(f"RAW_DIR exists: {os.path.exists(RAW_DIR)}")
print("导入完成 ✓")

BASE_DIR: c:\Users\admin\Desktop\newdaba
RAW_DIR exists: True
导入完成 ✓


In [2]:
# ============================================================
# 工具函数
# ============================================================
def scan_stocks(raw_dir: str) -> list:
    """扫描所有股票目录，返回股票代码列表"""
    stocks = []
    if not os.path.exists(raw_dir):
        print(f"[错误] 原始数据目录不存在: {raw_dir}")
        return stocks
    for name in os.listdir(raw_dir):
        stock_dir = os.path.join(raw_dir, name)
        if os.path.isdir(stock_dir):
            has_data = any(
                os.path.exists(os.path.join(stock_dir, f))
                for f in ['行情.csv', '逐笔成交.csv', '逐笔委托.csv']
            )
            if has_data:
                stocks.append(name)
    stocks.sort()
    return stocks


def format_duration(seconds: float) -> str:
    """格式化耗时"""
    if seconds < 60:
        return f"{seconds:.1f}秒"
    elif seconds < 3600:
        return f"{seconds / 60:.1f}分"
    else:
        return f"{seconds / 3600:.1f}小时"

print("工具函数就绪 ✓")

工具函数就绪 ✓


## 【1/4】阶段1：扫描股票列表

In [3]:
t_start = time.time()

stocks = scan_stocks(RAW_DIR)
if not stocks:
    print("[错误] 未找到任何股票数据，请检查 data/raw/100stock/ 目录")
else:
    print(f"发现 {len(stocks)} 只股票待处理")
    print(f"前5只: {stocks[:5]}")
    print(f"后5只: {stocks[-5:] if len(stocks) > 5 else stocks}")

发现 100 只股票待处理
前5只: ['600009.SH', '600018.SH', '600025.SH', '600076.SH', '600127.SH']
后5只: ['688082.SH', '688169.SH', '688496.SH', '688509.SH', '688560.SH']


## 【2/4】阶段2：逐只股票特征提取 → data/features/

⚠️ 本单元格运行时间较长，请耐心等待。

In [4]:
if not stocks:
    print("[跳过] 无股票数据")
else:
    os.makedirs(FEATURES_DIR, exist_ok=True)

    success_count = 0
    fail_list = []

    for i, stock_code in enumerate(stocks, 1):
        t_stock_start = time.time()
        try:
            # 2a. 加载并预处理
            data = load_and_preprocess_stock(stock_code, RAW_DIR)
            if data.get('quotes') is None and data.get('trades') is None:
                print(f"  [{i:3d}/{len(stocks)}] {stock_code} — 无有效数据，跳过")
                fail_list.append(stock_code)
                continue

            # 2b. 提取特征
            features = extract_all_features(stock_code, data)

            # 2c. 即时保存
            save_features(stock_code, features, FEATURES_DIR)

            t_stock = time.time() - t_stock_start
            print(f"  [{i:3d}/{len(stocks)}] {stock_code} ✓ ({t_stock:.1f}s)"
                  f" | 日期={features.get('transaction_date','?')}"
                  f" | 特征={len(features)-2}维")
            success_count += 1

        except Exception as e:
            print(f"  [{i:3d}/{len(stocks)}] {stock_code} ✗ 失败: {e}")
            fail_list.append(stock_code)

    print(f"\n特征提取完成: 成功 {success_count}/{len(stocks)} 只")
    if fail_list:
        print(f"失败列表: {fail_list}")

  [  1/100] 600009.SH ✓ (0.2s) | 日期=20260610 | 特征=60维
  [  2/100] 600018.SH ✓ (0.3s) | 日期=20260610 | 特征=60维
  [  3/100] 600025.SH ✓ (0.3s) | 日期=20260610 | 特征=60维
  [  4/100] 600076.SH ✓ (0.2s) | 日期=20260610 | 特征=60维
  [  5/100] 600127.SH ✓ (0.1s) | 日期=20260610 | 特征=60维
  [  6/100] 600152.SH ✓ (0.3s) | 日期=20260610 | 特征=60维
  [  7/100] 600156.SH ✓ (0.2s) | 日期=20260610 | 特征=60维
  [  8/100] 600161.SH ✓ (0.2s) | 日期=20260610 | 特征=60维
  [  9/100] 600172.SH ✓ (1.2s) | 日期=20260610 | 特征=60维
  [ 10/100] 600186.SH ✓ (1.4s) | 日期=20260610 | 特征=60维
  [ 11/100] 600203.SH ✓ (0.3s) | 日期=20260610 | 特征=60维
  [ 12/100] 600208.SH ✓ (0.3s) | 日期=20260610 | 特征=60维
  [ 13/100] 600293.SH ✓ (0.4s) | 日期=20260610 | 特征=60维
  [ 14/100] 600303.SH ✓ (0.2s) | 日期=20260610 | 特征=60维
  [ 15/100] 600330.SH ✓ (1.2s) | 日期=20260610 | 特征=60维
  [ 16/100] 600379.SH ✓ (0.2s) | 日期=20260610 | 特征=60维
  [ 17/100] 600382.SH ✓ (0.2s) | 日期=20260610 | 特征=60维
  [ 18/100] 600396.SH ✓ (2.3s) | 日期=20260610 | 特征=60维
  [ 19/100] 600448.SH ✓ (0.1

## 【3/4】阶段3：加载全量特征

In [5]:
df_features = load_all_features(FEATURES_DIR)
print(f"加载特征矩阵: {df_features.shape[0]} 样本 × {df_features.shape[1]} 维特征")

if df_features.empty:
    print("[错误] 特征矩阵为空，请先执行阶段2")
else:
    print(f"前5列: {list(df_features.columns[:5])}")
    print(f"后5列: {list(df_features.columns[-5:])}")

加载特征矩阵: 100 样本 × 62 维特征
前5列: ['stock_code', 'transaction_date', 'oss_mega_amount_pct', 'oss_mega_count_pct', 'oss_large_amount_pct']
后5列: ['cb_cancel_amount_ratio', 'cb_cancel_interval_cv', 'cb_fast_cancel_ratio', 'cb_buy_cancel_ratio', 'cb_sell_cancel_ratio']


### Task1：交易模式聚类（无监督 KMeans）

In [6]:
if df_features.empty:
    print("[跳过] 无特征数据")
else:
    print("--- Task1: 交易模式聚类 ---")
    df_features, X_scaled, feature_cols = run_clustering(df_features)
    df_features = assign_patterns(df_features, feature_cols)

    # 保存 Task1 结果
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    pattern_path = os.path.join(OUTPUT_DIR, 'pattern_reco.csv')
    save_pattern_result(df_features, pattern_path)

    # 输出模式分布
    print(f"\n交易模式分布:")
    dist = df_features['pattern_type'].value_counts()
    for pattern, cnt in dist.items():
        print(f"  {pattern}: {cnt} 只")

--- Task1: 交易模式聚类 ---
  [聚类评估] 轮廓系数: 0.1418 | CH指数: 11.77 | DB指数: 1.3900
  [聚类] 8 个聚类 | 各簇样本数:
    簇0: 1 只股票
    簇1: 45 只股票
    簇2: 17 只股票
    簇3: 13 只股票
    簇4: 15 只股票
    簇5: 2 只股票
    簇6: 3 只股票
    簇7: 4 只股票
  [保存] pattern_reco.csv → c:\Users\admin\Desktop\newdaba\output\pattern_reco.csv

交易模式分布:
  游资对倒出货: 82 只
  机构长线配置: 15 只
  游资强势连板拉升: 3 只


### Task2：资金类型与交易意图识别（三分类：游资/量化机构/散户）

In [7]:
if df_features.empty:
    print("[跳过] 无特征数据")
else:
    print("--- Task2: 资金与意图识别 ---")
    df_features = identify_all(df_features)

    # 保存 Task2 结果
    predict_path = os.path.join(OUTPUT_DIR, 'predict_result.csv')
    save_predict_result(df_features, predict_path)

--- Task2: 资金与意图识别 ---
  [Task2资金类型] 游资: 18 | 量化机构: 54 | 散户: 28
    游资占比: 18.0% | 量化占比: 54.0% | 散户占比: 28.0%
  [Task2交易意图] 买入: 0 | 卖出: 4 | T0交易: 96
    买入占比: 0.0% | 卖出占比: 4.0% | T0占比: 96.0%
  [保存] predict_result.csv → c:\Users\admin\Desktop\newdaba\output\predict_result.csv
  [校验] 格式合法 ✓


## 【4/4】阶段4：离线评估 & 交叉统计

In [8]:
if not df_features.empty:
    # 交叉统计1：资金类型 × 交易意图
    if 'capital_type' in df_features.columns and 'capital_intention' in df_features.columns:
        cross1 = pd.crosstab(df_features['capital_type'], df_features['capital_intention'])
        print("资金类型 × 交易意图 交叉表:")
        print(cross1.to_string())
    else:
        print("[警告] 缺少 capital_type 或 capital_intention 列，请先执行 Task2")

    # 交叉统计2：交易模式 × 资金类型
    if 'pattern_type' in df_features.columns and 'capital_type' in df_features.columns:
        cross2 = pd.crosstab(df_features['pattern_type'], df_features['capital_type'])
        print(f"\n交易模式 × 资金类型 交叉表:")
        print(cross2.to_string())
else:
    print("[跳过] 无数据")

资金类型 × 交易意图 交叉表:
capital_intention  T0交易  卖出
capital_type               
散户                   26   2
游资                   16   2
量化机构                 54   0

交易模式 × 资金类型 交叉表:
capital_type  散户  游资  量化机构
pattern_type              
机构长线配置         4   0    11
游资对倒出货        23  18    41
游资强势连板拉升       1   0     2


## ✅ 全流程完成

```python
# 总耗时
t_total = time.time() - t_start
print(f"总耗时: {format_duration(t_total)}")
print(f"输出文件:")
print(f"  {os.path.join(OUTPUT_DIR, 'pattern_reco.csv')}")
print(f"  {os.path.join(OUTPUT_DIR, 'predict_result.csv')}")
print(f"特征文件: {FEATURES_DIR}/")
```

### 各阶段独立性

| 单元格 | 说明 | 是否可独立运行 |
|--------|------|:---:|
| 1 | 导入 & 路径 | ✅ 始终运行 |
| 2 | 工具函数 | ✅ 依赖单元格1 |
| 3 | 阶段1 扫描 | ✅ 依赖单元格1-2 |
| 4 | 阶段2 特征提取 | ✅ 依赖单元格1-3 |
| 5 | 阶段3 加载特征 | ✅ 依赖单元格4 |
| 6 | Task1 聚类 | ✅ 依赖单元格5 |
| 7 | Task2 意图（游资/量化机构/散户三分类） | ✅ 依赖单元格5（与Task1并行） |
| 8 | 阶段4 评估 | ✅ 依赖单元格6-7 |

### 格式校验（save_predict_result 自动执行）
- capital_type 合法值：`游资` / `量化机构` / `散户`
- capital_intention 合法值：`买入` / `卖出` / `T0交易`